In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter
from pathlib import PurePosixPath
import subprocess
import sys
import json

from collections import defaultdict
from io import BytesIO

from IPython.display import display
from PIL import Image, ImageDraw

In [3]:
archive_dir = Path("/content/drive/MyDrive/WasteScope/raw_archives")

archives = {
    "detection": archive_dir / "detection_task.zip",
    "tracking": archive_dir / "tracking_task.zip",
    "segmentation": archive_dir / "segmentation_task.zip",
}

In [4]:
def video_id(filename):
    return PurePosixPath(filename).stem.split("_")[0]

image_counts = {}

for task, archive in archives.items():
    with ZipFile(archive) as zf:
        image_counts[task] = Counter(
            video_id(item.filename)
            for item in zf.infolist()
            if item.filename.lower().endswith(".png")
        )

with ZipFile(archives["segmentation"]) as zf:
    with zf.open("Segmentation Task/labels.json") as file:
        coco = json.load(file)

category_names = {
    category["id"]: category["name"]
    for category in coco["categories"]
}

image_to_video = {
    image["id"]: video_id(image["file_name"])
    for image in coco["images"]
}

In [5]:
annotated_image_ids = set()
category_instances = defaultdict(Counter)
category_images = defaultdict(lambda: defaultdict(set))

for annotation in coco["annotations"]:
    image_id = annotation["image_id"]
    category_id = annotation["category_id"]
    
    vid = image_to_video[image_id]
    annotated_image_ids.add(image_id)
    category_instances[vid][category_id] += 1
    category_images[vid][category_id].add(image_id)
    
empty_images = Counter(
    video_id(image["file_name"])
    for image in coco["images"]
    if image["id"] not in annotated_image_ids
)

print("Category ID legend:")
for category_id, name in sorted(category_names.items()):
    print(f"  {category_id}: {name}")

Category ID legend:
  1: Overflow
  2: container_yellow
  3: container_green
  4: container_default
  5: container_blue
  6: container_ash
  7: container_oil
  8: container_battery
  9: container_biodegradable


In [6]:
print("Per-video distribution (segmentation 38 video):")
print("video   detection  tracking  segmentation  overflow+  empty  categories")

for vid in sorted(image_counts["segmentation"]):
    present_categories = ",".join(
        str(category_id)
        for category_id in sorted(category_names)
        if category_instances[vid][category_id] > 0
    )

    print(
        f"{vid:6}  "
        f"{image_counts['detection'][vid]:9}  "
        f"{image_counts['tracking'][vid]:8}  "
        f"{image_counts['segmentation'][vid]:12}  "
        f"{len(category_images[vid][1]):9}  "
        f"{empty_images[vid]:5}  "
        f"{present_categories}"
    )

Per-video distribution (segmentation 38 video):
video   detection  tracking  segmentation  overflow+  empty  categories
0002           50         0            50         50      0  1,4
0003          290         0           290        290      0  1,4
0008          140         0           140          0      1  4
0012          260         0           260        260      0  1,2,3,4,5,8
0018          340        90           250        107     13  1,2,3,5
0019          440         0           440          0      0  2,3,4,5,9
0020          600         0           600        505      0  1,2,3,4,5,8,9
0023          210         0           210        142      0  1,2,3,4,5,9
0025          200         0           200          0      0  2,3,4,5,8,9
0031          170         0           170         78     19  1,2,3,4,5,7,8
0034          120         0           120         69      0  1,2,3,4,5
0035          150       100            50          0      0  2,3,5
0038           90         0            9

In [7]:
print("Category coverage across videos:")
print("ID  category videos  images  instances:")

for category_id, name in sorted(category_names.items()):
    videos = [
        vid for vid in image_counts["segmentation"]
        if category_instances[vid][category_id] > 0
    ]
    images = sum(
        len(category_images[vid][category_id])
        for vid in videos
    )
    instances = sum(
        category_instances[vid][category_id]
        for vid in videos
    )

    print(
        f"{category_id:2}  {name:24} "
        f"{len(videos):6}  {images:6}  {instances:9}"
    )

    if len(videos) <= 10:
        print("Video IDs:", ", ".join(sorted(videos)))

Category coverage across videos:
ID  category videos  images  instances:
 1  Overflow                     33    4197       5149
 2  container_yellow             32    4933       5191
 3  container_green              32    4909       4909
 4  container_default            34    5700      12349
 5  container_blue               32    5144       5448
 6  container_ash                 2     212        212
Video IDs: 0054, 0110
 7  container_oil                 7     983        983
Video IDs: 0031, 0038, 0047, 0048, 0052, 0107, 0110
 8  container_battery            11    1247       1301
 9  container_biodegradable       5    1212       1212
Video IDs: 0019, 0020, 0023, 0025, 0047
